# GPU part (Colab)

Runs VGGT on both videos of a pair and saves `cache/<pair>.npz` for the Mac.
Runtime > Change runtime type > T4 GPU, then run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "https://github.com/golovchits/visibility-aware-change-detection.git"
!git clone -q {REPO} /content/repo || git -C /content/repo pull -q
%cd /content/repo
# --no-deps: VGGT pins numpy<2, which would downgrade Colab's numpy. einops is the only missing dependency.
!pip install -q --no-deps git+https://github.com/facebookresearch/vggt.git
!pip install -q einops

## One joint VGGT run per pair
AB is the test pair, BC the no-change control.

In [ ]:
!python src/gpu.py --pair AB
!python src/gpu.py --pair BC

## Verify: one table, not two
Video 1 in blue, video 2 in orange. If the joint run worked, the colours mix on one table.
Only the more confident half of the points is shown here. The real cut-off is set on the Mac.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from vggt.utils.geometry import unproject_depth_map_to_point_map

def plot_pair(pair):
    cache = np.load(f"cache/{pair}.npz")
    # The VGGT helper expects depth as (frames, H, W, 1).
    points = unproject_depth_map_to_point_map(cache["depth"][..., None], cache["extrinsic"], cache["intrinsic"])
    video = np.broadcast_to(cache["video"][:, None, None], cache["depth"].shape)

    keep = cache["confidence"] > np.median(cache["confidence"])
    points, video = points[keep], video[keep]
    pick = np.random.default_rng(0).choice(len(points), size=min(100_000, len(points)), replace=False)
    points, video = points[pick], video[pick]

    fig = go.Figure()
    for tag, colour in [(1, "royalblue"), (2, "darkorange")]:
        p = points[video == tag]
        fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
                                   marker=dict(size=1, color=colour), name=f"{pair[int(tag) - 1]} (video {tag})"))
    fig.update_layout(title=pair, scene=dict(aspectmode="data"), height=700)
    fig.show()

plot_pair("AB")
plot_pair("BC")

## Download the caches
Put both files in `cache/` on the Mac.

In [ ]:
from google.colab import files
files.download("cache/AB.npz")
files.download("cache/BC.npz")